In [1]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [2]:
# Source and destination dataset directories
source_root = "../OnderzoekDataSubset"
destination_root = "../Yolo/OnderzoekDataSubsetYolo"

In [3]:
# Create directory structure
for split in ["Train", "Test"]:
    os.makedirs(os.path.join(destination_root, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(destination_root, split, "labels"), exist_ok=True)

In [4]:
# Helper function to generate YOLO annotation from a mask
def generate_yolo_annotations(image, mask):
    height, width = mask.shape

    # Ensure binary mask
    mask = mask * 255
    mask = mask.astype(np.uint8)
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []

    yolo_annotations = []
    for contour in contours:
        if len(contour) < 3:
            continue  # Skip invalid polygons

        polygon_normalized = [(x / width, y / height) for [[x, y]] in contour]
        annotation_line = "0 " + " ".join([f"{x:.6f} {y:.6f}" for x, y in polygon_normalized])
        yolo_annotations.append(annotation_line)

    return yolo_annotations

In [5]:
# Process both splits
for split in ["Train", "Test"]:
    src_dir = Path(source_root) / split
    dst_images = Path(destination_root) / split.lower() / "images"
    dst_labels = Path(destination_root) / split.lower() / "labels"

    # Collect all mask-image pairs
    for item in tqdm(os.listdir(src_dir), desc=f"Processing {split}"):
        item_path = src_dir / item
        if item.endswith("mask.png"):
            image_path = str(item_path).replace("mask.png", "image.png")
            image_name = os.path.basename(image_path)
            label_name = image_name.replace(".png", ".txt")

            # Load image and mask
            image = cv2.imread(image_path)
            mask = cv2.imread(str(item_path), cv2.IMREAD_GRAYSCALE)

            if image is None or mask is None:
                print(f"Skipping {image_path}, file not found or corrupted.")
                continue

            # Generate annotations and write them to the dataset
            annotations = generate_yolo_annotations(image, mask)
            cv2.imwrite(str(dst_images / image_name), image)
            with open(dst_labels / label_name, "w") as f:
                f.write("\n".join(annotations))


Processing Test: 100%|██████████| 600/600 [00:05<00:00, 114.13it/s]
